# Импорты

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import warnings
import os
import random
from torch.cuda.amp import autocast, GradScaler
warnings.filterwarnings('ignore')

In [2]:
def set_seed(seed=42):
    
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    np.random.seed(seed)
    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    print(f"✓ Seed set to {seed} (deterministic mode)")

# Метрика

In [3]:
def wape_plus_rbias(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    s = y_true.sum()
    if s == 0:
        return 0.0
    return np.abs(y_pred - y_true).sum() / s + np.abs(y_pred.sum() / s - 1)

# Загрузка данных

In [4]:
train = pd.read_parquet('train_team_track.parquet')
test = pd.read_parquet('test_team_track.parquet')
train['timestamp'] = pd.to_datetime(train['timestamp'])
test['timestamp'] = pd.to_datetime(test['timestamp'])

route_to_office = train.groupby('route_id')['office_from_id'].first().to_dict()
test['office_from_id'] = test['route_id'].map(route_to_office)

status_cols = [f'status_{i}' for i in range(1, 9)]
status_cols =  [col for col in status_cols if col not in ['status_1', 'status_3']]
for c in status_cols:
    test[c] = np.nan

WINDOW_SIZE = 48
FORECAST_HORIZON = 10

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
torch.cuda.manual_seed(seed)
set_seed()
        
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

✓ Seed set to 42 (deterministic mode)
Device: cuda


# Конфигурации

In [5]:
MODE = 'final'

configs = {
    'experiment': {
        'sample_routes': 200,
        'epochs': 12,
        'patience': 4,
        'batch_size': 2048
    },
    'validation': {
        'sample_routes': 500,
        'epochs': 20,
        'patience': 5,
        'batch_size': 2048
    },
    'final': {
        'sample_routes': None,
        'epochs': 70,
        'patience': 10,
        'batch_size': 2048
    }
}

cfg = configs[MODE]
print(f"MODE: {MODE}")
print(f"Config: {cfg}")

EPOCHS = cfg['epochs']
PATIENCE = cfg['patience']
BS = cfg['batch_size']

MODE: final
Config: {'sample_routes': None, 'epochs': 70, 'patience': 10, 'batch_size': 2048}


### Плохие без невозможных

In [6]:
bad_routes = [
    243, 705, 385, 602, 823, 77, 939, 844, 406, 190, 629, 936, 43, 420, 555,
    31, 326, 386, 966, 124, 268, 455, 71, 370, 703, 790, 215, 221, 882, 9,
    316, 407, 897, 920, 577, 253, 977, 28, 248, 519, 498, 637, 307, 928, 319,
    539, 250, 328, 185, 874, 361, 364, 464, 838, 911, 106, 301, 557, 659, 48,
    324, 528, 471, 746, 383, 440, 454, 921, 429, 846, 431, 776, 280, 390, 668,
    775, 803, 829, 265, 244, 931, 134, 245, 359, 604, 603, 172, 613, 78, 672,
    125, 679, 860, 791, 505, 626, 21, 809, 877, 542, 81, 195, 436, 811, 38,
    350, 42, 709, 851, 338, 291, 300, 616, 617, 387, 793, 717, 40, 282, 147,
    728, 837, 16, 956, 45, 531, 739, 474, 270, 278, 367, 0, 635, 463, 126, 889,
    50, 638, 902, 136, 417, 768, 450, 595, 187, 845, 562, 413, 912, 704, 87,
    58, 320, 854, 968, 796, 959, 150, 574, 346, 546, 446, 712, 946, 170, 236,
    675, 738, 304, 678, 5, 896, 96, 377, 849, 461, 252, 556, 500, 644, 870,
    563, 517, 927, 131, 581, 850, 951, 590, 225, 682, 220, 381, 3, 382, 355,
    141, 373, 74, 322, 62, 480, 906, 437, 514, 591, 35, 560, 371, 80, 518, 884,
    162, 689, 869, 525, 875, 360, 495, 102, 426, 745, 727, 308, 687, 916, 69,
    733, 864, 601, 135, 881, 122, 651, 493, 609, 723, 151, 582, 685, 284, 70,
    848, 918, 862, 566, 569, 633, 969, 204, 647, 527, 267, 357, 526, 22, 188,
    262, 789, 397, 891, 901, 804, 841, 996, 598, 273, 388, 179, 639, 175, 506,
    286, 137, 46, 340
]
very_bad_routes = [243, 124, 370, 215, 407, 977, 928, 364, 931, 505, 21, 531, 474, 450, 968, 582, 304, 518]
# other_bad = [42,87,126,136,328,377,413,431,437,454,560,569,577,609,651,678,775,811,846,874,875,902,912,916, 927, 304, 518]

available_routes = np.setdiff1d(bad_routes, very_bad_routes)
# available_routes = np.setdiff1d(available_routes, other_bad)

# sample_routes = np.random.choice(
#     available_routes,
#     size=cfg['sample_routes'],
#     replace=False
# )
train = train[train['route_id'].isin(available_routes)].copy()
train = train.reset_index(drop=True)

# Скейлеры

In [7]:
class LogRobustScaler:
    def __init__(self):
        self.medians = {}
        self.iqrs = {}
        self.use_log = {}

    def fit(self, df, columns, log_columns=None):
        log_columns = log_columns or []
        for col in columns:
            v = df[col].dropna().values.copy()
            if col in log_columns:
                v = np.log1p(np.clip(v, 0, None))
                self.use_log[col] = True
            else:
                self.use_log[col] = False
            q25, q75 = np.percentile(v, [25, 75])
            self.medians[col] = np.median(v)
            self.iqrs[col] = max(q75 - q25, 1e-4)
        return self

    def transform(self, values, col):
        v = np.array(values, dtype=float)
        if self.use_log.get(col, False):
            v = np.log1p(np.clip(v, 0, None))
        return (v - self.medians[col]) / self.iqrs[col]

    def inverse_transform(self, values, col):
        v = np.array(values, dtype=float)
        v = v * self.iqrs[col] + self.medians[col]
        if self.use_log.get(col, False):
            v = np.expm1(v)
        return v

In [8]:
log_cols = status_cols
scaler = LogRobustScaler()
scaler.fit(train, log_cols, log_columns=log_cols)

target_scaler = LogRobustScaler()
target_scaler.fit(train, ['target_2h'])

# Создание фичей

In [9]:
def add_time_features(df):
    df = df.copy()
    df['time_slot'] = df['timestamp'].dt.hour * 2 + df['timestamp'].dt.minute // 30
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['hour'] = df['timestamp'].dt.hour + df['timestamp'].dt.minute/60
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(float)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['slot_sin'] = np.sin(2 * np.pi * df['time_slot'] / 48)
    df['slot_cos'] = np.cos(2 * np.pi * df['time_slot'] / 48)
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    return df

train = add_time_features(train)
test = add_time_features(test)

In [10]:
time_features = ['hour_sin', 'hour_cos', 'slot_sin', 'slot_cos',
                 'dow_sin', 'dow_cos', 'is_weekend']

seq_cols = status_cols + ['target_2h']
num_seq_features = len(seq_cols) + len(time_features)

# Построение окон

### Дефолтное

In [11]:
print("Building sequences efficiently...")

train_sorted = train

for c in status_cols:
    train_sorted[f'{c}_norm'] = scaler.transform(train_sorted[c].values, c)
train_sorted['target_2h_norm'] = target_scaler.transform(train_sorted['target_2h'].values, 'target_2h')

train_sorted = train_sorted.sort_values(['route_id', 'timestamp']).reset_index(drop=True)

all_X = []
all_y = []
all_rids = []
all_oids = []
all_ts = []

norm_cols = [f'{c}_norm' for c in status_cols]
feature_matrix = train_sorted[norm_cols + time_features + ['target_2h_norm']].values
targets = train_sorted['target_2h_norm'].values

route_groups = train_sorted.groupby('route_id', sort=False)

for rid, group_indices in route_groups.groups.items():
    indices = group_indices.values
    n_rows = len(indices)
    
    if n_rows <= WINDOW_SIZE + FORECAST_HORIZON:
        continue
    
    route_features = feature_matrix[indices]
    route_target = targets[indices]
    route_ts = train_sorted.loc[indices, 'timestamp'].values
    route_oid = train_sorted.loc[indices[0], 'office_from_id']
    
    for i in range(WINDOW_SIZE, n_rows - FORECAST_HORIZON + 1):
        all_X.append(route_features[i - WINDOW_SIZE:i])
        all_y.append(route_target[i:i + FORECAST_HORIZON])
        all_rids.append(rid)
        all_oids.append(route_oid)
        all_ts.append(route_ts[i])

all_X = np.array(all_X, dtype=np.float32)
all_y = np.array(all_y, dtype=np.float32)
all_rids = np.array(all_rids, dtype=np.int16)
all_oids = np.array(all_oids, dtype=np.int16)
all_ts = np.array(all_ts)

print(f"Sequences: {all_X.shape}, Targets: {all_y.shape}")

Building sequences efficiently...
Sequences: (1105530, 48, 14), Targets: (1105530, 10)


# Датасеты и даталоэдеры

In [12]:
last_10_timestamps = sorted(train['timestamp'].unique())[-10:]
last_10_timestamps_np = pd.to_datetime(last_10_timestamps).to_numpy()
print(f"\nValidation timestamps (last 10): {last_10_timestamps[0]} — {last_10_timestamps[-1]}")

val_mask = np.isin(all_ts, last_10_timestamps_np)
train_mask = ~val_mask

X_tr, y_tr = all_X[train_mask], all_y[train_mask]
r_tr, o_tr = all_rids[train_mask], all_oids[train_mask]
X_va, y_va = all_X[val_mask], all_y[val_mask]
r_va, o_va = all_rids[val_mask], all_oids[val_mask]

print(f"Train: {X_tr.shape[0]:,}, Val: {X_va.shape[0]:,}")


Validation timestamps (last 10): 2025-05-30 06:00:00 — 2025-05-30 10:30:00
Train: 1,105,272, Val: 258


In [13]:
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 32

train_ds_sparse = TensorDataset(
    torch.tensor(X_tr, dtype=torch.float32),
    torch.tensor(r_tr, dtype=torch.long),
    torch.tensor(o_tr, dtype=torch.long),
    torch.tensor(y_tr, dtype=torch.float32)
)

val_ds_sparse = TensorDataset(
    torch.tensor(X_va, dtype=torch.float32),
    torch.tensor(r_va, dtype=torch.long),
    torch.tensor(o_va, dtype=torch.long),
    torch.tensor(y_va, dtype=torch.float32)
)

train_loader_sparse = DataLoader(train_ds_sparse, batch_size=BATCH_SIZE, shuffle=True)
val_loader_sparse = DataLoader(val_ds_sparse, batch_size=BATCH_SIZE, shuffle=False)

# Модель

In [20]:
class SimpleModelForSparse(nn.Module):
    """
    Упрощённая модель для разреженных маршрутов.
    X содержит: [status_2, status_4, status_5, status_6, status_7, status_8, 
                 time_features(7), target_2h_norm]
    """
    def __init__(self, nf=6, n_time_features=7, nr=1000, no=54,
                 re=32, oe=16, hidden=64, forecast_horizon=10):
        super().__init__()
        
        self.nf = nf
        self.n_time_features = n_time_features
        self.forecast_horizon = forecast_horizon
        
        # Embeddings для маршрута и офиса
        self.r_emb = nn.Embedding(nr, re)
        self.o_emb = nn.Embedding(no, oe)
        
        # Входные размеры:
        # - nf статусов (усреднённых) = 6
        # - n_time_features time features = 7
        # - embeddings = re + oe = 32 + 16 = 48
        # Итого: 6 + 7 + 48 = 61
        head_in = nf + n_time_features + re + oe 
        
        self.head = nn.Sequential(
            nn.Linear(head_in, hidden), 
            nn.BatchNorm1d(hidden), 
            nn.GELU(), 
            nn.Dropout(0.3),
            nn.Linear(hidden, hidden // 2), 
            nn.BatchNorm1d(hidden // 2), 
            nn.GELU(), 
            nn.Dropout(0.2),
            nn.Linear(hidden // 2, forecast_horizon)
        )
    
    def forward(self, x, rid, oid):
        """
        x: [batch, window_size, 14]
           - первые 6 элементов: статусы
           - следующие 7 элементов: time features
           - последний элемент: target_2h_norm (игнорируем)
        rid, oid: [batch]
        """
        batch_size = x.shape[0]
        
        # Раздели компоненты X
        # [status(6) | time_features(7) | target(1)]
        x_status = x[:, :, :self.nf]  # [batch, window_size, 6]
        x_time = x[:, :, self.nf:self.nf + self.n_time_features]  # [batch, window_size, 7]
        # x_target = x[:, :, -1]  # Не используем (последний элемент)
        
        # Усредни статусы по времени
        x_status_mean = x_status.mean(dim=1)  # [batch, 6]
        
        # Возьми time features из последней позиции окна
        x_time_last = x_time[:, -1, :]  # [batch, 7]
        
        # Embeddings
        r = self.r_emb(rid)  # [batch, 32]
        o = self.o_emb(oid)  # [batch, 16]
        
        # Объедини всё: 6 + 7 + 32 + 16 = 61
        features = torch.cat([x_status_mean, x_time_last, r, o], dim=1)
        
        # Предсказание
        return self.head(features)

In [21]:
n_time_features = len(time_features)  # 7

model_sparse = SimpleModelForSparse(
    nf=6,  # status_2, status_4, status_5, status_6, status_7, status_8
    n_time_features=n_time_features,  # 7 time features
    nr=1000,
    no=54,
    re=32,
    oe=16,
    hidden=64,
    forecast_horizon=10
)
model_sparse = model_sparse.to(device)

In [ ]:
model.load_state_dict(torch.load('model_weights_sparse_all.pth'))

# Функции ошибки

In [22]:
optimizer_sparse = torch.optim.AdamW(
    model_sparse.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

# Scheduler
scheduler_sparse = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_sparse,
    mode='min',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=True
)

# Функция потерь
criterion_sparse = nn.MSELoss()  # Для разреженных используем просто MSE

In [23]:
best_metric, best_epoch, best_state = float('inf'), 0, None
checkpoint_path = 'best_model.pth'

# Обучение

### Дефолтная тренировка

In [25]:
EPOCHS = 100
PATIENCE = 20

best_metric_sparse = float('inf')
best_epoch_sparse = 0
patience_counter = 0

print("\n" + "="*70)
print("ОБУЧЕНИЕ МОДЕЛИ ДЛЯ РАЗРЕЖЕННЫХ МАРШРУТОВ")
print("="*70)

for ep in range(EPOCHS):
    # ===================== ОБУЧЕНИЕ =====================
    model_sparse.train()
    losses = []
    
    for xb, rb, ob, yb in train_loader_sparse:
        xb, rb, ob, yb = [t.to(device) for t in [xb, rb, ob, yb]]
        
        optimizer_sparse.zero_grad()
        pred = model_sparse(xb, rb, ob)
        loss = criterion_sparse(pred, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model_sparse.parameters(), 1.0)
        optimizer_sparse.step()
        
        losses.append(loss.item())
    
    train_loss = np.mean(losses)
    
    # ===================== ВАЛИДАЦИЯ =====================
    model_sparse.eval()
    val_preds, val_trues = [], []
    
    with torch.no_grad():
        for xb, rb, ob, yb in val_loader_sparse:
            xb, rb, ob = [t.to(device) for t in [xb, rb, ob]]
            pred = model_sparse(xb, rb, ob).cpu().numpy()
            val_preds.append(pred)
            val_trues.append(yb.numpy())
    
    val_preds = np.concatenate(val_preds)  # [N, 10]
    val_trues = np.concatenate(val_trues)  # [N, 10]
    
    # Денормализуй для метрики
    val_preds_denorm = np.clip(
        target_scaler.inverse_transform(val_preds.flatten(), 'target_2h'), 
        0, None
    ).reshape(val_preds.shape)
    val_trues_denorm = target_scaler.inverse_transform(
        val_trues.flatten(), 'target_2h'
    ).reshape(val_trues.shape)
    
    # Вычисли метрику
    val_metric = wape_plus_rbias(val_trues_denorm.flatten(), val_preds_denorm.flatten())
    
    # Scheduler step
    scheduler_sparse.step(val_metric)
    
    # Сохранение лучшей модели
    if val_metric < best_metric_sparse:
        best_metric_sparse = val_metric
        best_epoch_sparse = ep
        patience_counter = 0
        torch.save(model_sparse.state_dict(), f'best_model_sparse_ep{ep}.pth')
        print(f"  → Saved checkpoint: best_model_sparse_ep{ep}.pth")
    else:
        patience_counter += 1
    
    # Вывод результатов
    w = np.abs(val_preds_denorm - val_trues_denorm).sum() / val_trues_denorm.sum()
    b = np.abs(val_preds_denorm.sum() / val_trues_denorm.sum() - 1)
    
    print(f"Ep {ep:3d} | L {train_loss:.4f} | M {val_metric:.4f} (W:{w:.4f} B:{b:.4f})"
          f"{' *' if val_metric <= best_metric_sparse else ''}")
    
    # Early stopping
    if patience_counter >= PATIENCE:
        print(f"Early stop at epoch {ep}")
        break

print(f"\nОбучение завершено. Лучшая метрика: {best_metric_sparse:.4f} на эпохе {best_epoch_sparse}")


ОБУЧЕНИЕ МОДЕЛИ ДЛЯ РАЗРЕЖЕННЫХ МАРШРУТОВ
  → Saved checkpoint: best_model_sparse_ep0.pth
Ep   0 | L 0.4518 | M 0.5499 (W:0.5045 B:0.0454) *
Ep   1 | L 0.4479 | M 0.5634 (W:0.5054 B:0.0580)
  → Saved checkpoint: best_model_sparse_ep2.pth
Ep   2 | L 0.4451 | M 0.5321 (W:0.5015 B:0.0307) *
Ep   3 | L 0.4435 | M 0.5965 (W:0.5005 B:0.0959)
Ep   4 | L 0.4423 | M 0.5730 (W:0.5070 B:0.0660)
Ep   5 | L 0.4413 | M 0.5572 (W:0.5012 B:0.0559)
Ep   6 | L 0.4409 | M 0.5596 (W:0.5035 B:0.0561)


KeyboardInterrupt: 

In [ ]:
print(f"\nBest: ep {best_epoch}, metric {best_metric:.6f}")
model.load_state_dict(best_state)
model = model.to(device)

In [ ]:
checkpoint = torch.load('best_model_ep20.pth')
model.load_state_dict(checkpoint['model_state'])

In [ ]:
torch.save(model.state_dict(), 'model_weights_sparse_all.pth')

In [ ]:
model.eval()

test_preds, test_trues = [], []
with torch.no_grad():
    for xb, rb, ob, yb in val_loader:
        pred = model(
            torch.FloatTensor(xb).to(device), 
            torch.LongTensor(rb).to(device), 
            torch.LongTensor(ob).to(device)).cpu().numpy()
        test_preds.append(pred)
        test_trues.append(yb.numpy())

test_preds = np.concatenate(test_preds)  # [N, 10]
test_trues = np.concatenate(test_trues)  # [N, 10]

# Денормализуй
test_preds_denorm = np.clip(
    target_scaler.inverse_transform(test_preds.flatten(), 'target_2h'), 
    0, None
).reshape(test_preds.shape)
test_trues_denorm = target_scaler.inverse_transform(
    test_trues.flatten(), 'target_2h'
).reshape(test_trues.shape)
wape_plus_rbias(test_trues_denorm, test_preds_denorm)

In [ ]:
# Для каждого маршрута посчитай, насколько "нестабильный" таргет
route_volatility = {}
for rid in train['route_id'].unique():
    route_data = train[train['route_id'] == rid]['target_2h'].values
    if len(route_data) > 1:
        # Волатильность = стандартное отклонение / среднее
        mean_val = route_data.mean()
        if mean_val > 0:
            volatility = route_data.std() / mean_val
            route_volatility[rid] = volatility

# Разделим маршруты по волатильности
volatility_values = list(route_volatility.values())
low_vol_threshold = np.percentile(volatility_values, 33)
high_vol_threshold = np.percentile(volatility_values, 67)

low_vol_routes = [r for r, v in route_volatility.items() if v <= low_vol_threshold]
med_vol_routes = [r for r, v in route_volatility.items() if low_vol_threshold < v <= high_vol_threshold]
high_vol_routes = [r for r, v in route_volatility.items() if v > high_vol_threshold]

print(f"Low volatility маршруты: {len(low_vol_routes)}")
print(f"Medium volatility маршруты: {len(med_vol_routes)}")
print(f"High volatility маршруты: {len(high_vol_routes)}")

In [ ]:
model.eval()

route_metrics = {}

with torch.no_grad():
    for xb, rb, ob, yb in val_loader:
        pred = model(
            torch.FloatTensor(xb).to(device), 
            torch.LongTensor(rb).to(device), 
            torch.LongTensor(ob).to(device)).cpu().numpy()
        
        pred_d = np.clip(target_scaler.inverse_transform(pred.flatten(), 'target_2h'), 0, None).reshape(pred.shape)
        true_d = target_scaler.inverse_transform(yb.numpy().flatten(), 'target_2h').reshape(yb.numpy().shape)
        
        for i, rid in enumerate(rb.numpy()):
            if rid not in route_metrics:
                route_metrics[rid] = {'preds': [], 'trues': []}
            route_metrics[rid]['preds'].append(pred_d[i])
            route_metrics[rid]['trues'].append(true_d[i])

# Посчитай метрику для каждого маршрута
route_errors = {}
for rid in route_metrics:
    all_preds = np.concatenate(route_metrics[rid]['preds'])
    all_trues = np.concatenate(route_metrics[rid]['trues'])
    metric = wape_plus_rbias(all_trues, all_preds)
    route_errors[rid] = metric

# Топ лучших и худших маршрутов
print("ТОП-20 ЛУЧШИХ маршрутов:")
for rid, err in sorted(route_errors.items(), key=lambda x: x[1])[:20]:
    print(f"  Route {rid:3d}: {err:.4f}")

print("\nТОП-20 ХУДШИХ маршрутов:")
for rid, err in sorted(route_errors.items(), key=lambda x: x[1], reverse=True)[:20]:
    print(f"  Route {rid:3d}: {err:.4f}")

# Анализ: есть ли паттерн? (маршруты с низким таргетом? высоким?)
print("\nАнализ худших маршрутов:")
worst_20_routes = [r for r, _ in sorted(route_errors.items(), key=lambda x: x[1], reverse=True)[:20]]
for rid in worst_20_routes:
    mean_target = train[train['route_id'] == rid]['target_2h'].mean()
    std_target = train[train['route_id'] == rid]['target_2h'].std()
    print(f"  Route {rid}: mean={mean_target:.2f}, std={std_target:.2f}, vol={std_target/mean_target if mean_target > 0 else 0:.2f}")

In [ ]:
train.groupby('route_id')['status_2'].mean()[lambda x: x == 0].index.tolist()

In [ ]:
route_means = train.groupby('route_id')['target_2h'].mean().to_dict()

ZERO_THRESHOLD = 5
zero_routes = [rid for rid, mean_val in route_means.items() if mean_val < ZERO_THRESHOLD]

print(f"Найдено {len(zero_routes)} 'нулевых' маршрутов:")
print(zero_routes)

In [ ]:
np.set_printoptions(threshold=np.inf)
print(train[train.route_id == 122]['target_2h'].values)
np.set_printoptions(threshold=20)

In [ ]:
train.groupby('route_id')[['status_2', 'status_4']].mean().query('status_2 < 1 and status_4 < 1').index.tolist()

In [ ]:
train.groupby('route_id')['status_4'].mean()[lambda x: x == 0].index.tolist()

In [ ]:
# Анализ проблемных маршрутов
pd.set_option('display.max_seq_items', None)
np.set_printoptions(threshold=np.inf)
problem_routes = [300, 147]

for rid in problem_routes:
    route_data = train[train['route_id'] == rid]
    print(f"\n{'='*70}")
    print(f"Route {rid}")
    print(f"{'='*70}")
    print(f"Количество записей: {len(route_data)}")
    print(f"Mean target: {route_data['target_2h'].mean():.2f}")
    print(f"Std target: {route_data['target_2h'].std():.2f}")
    print(f"Min: {route_data['target_2h'].min()}")
    print(f"Max: {route_data['target_2h'].max()}")
    print(f"\nВсе значения таргета для этого маршрута:")
    print(route_data['target_2h'].values)
    print(f"\nТекущие статусы:")
    for col in status_cols:
        print(f"  {col}: mean={route_data[col].mean():.2f}, std={route_data[col].std():.2f}")
pd.set_option('display.max_seq_items', 20)
np.set_printoptions(threshold=20)

# Предсказание тестовых данных

### Дефолтное

In [ ]:
test = test[test['route_id'].isin(available_routes)]

In [ ]:
test = train_sorted.groupby('route_id').tail(10)

In [ ]:
test = test.reset_index().rename(columns={'index':'id'})

In [ ]:
train_sorted = train_sorted[train_sorted.groupby('route_id', sort=False).cumcount().between(4341-WINDOW_SIZE-10, 4341-11)]

In [ ]:
route_histories = {}

for rid in train_sorted['route_id'].unique():
    rd = train_sorted[train_sorted['route_id'] == rid]
    norm_parts = [scaler.transform(rd[c].values, c) for c in status_cols]
    norm_arr = np.column_stack(norm_parts)
    time_arr = rd[time_features].values
    target_arr = rd['target_2h_norm'].values.reshape(-1, 1)
    full = np.column_stack([norm_arr, time_arr, target_arr]).astype(np.float32)
    route_histories[rid] = list(full[-WINDOW_SIZE:])

In [ ]:
test_by_route = test.groupby('route_id', sort=False)

id_to_pred = {}

model.eval()

print("\nInference on test set...")
for rid, group_df in test_by_route:    
    oid = int(group_df.iloc[0]['office_from_id'])
    batch_ids = group_df['id'].values
    
    hist = np.array(route_histories[rid], dtype=np.float32)  # [WINDOW_SIZE, num_features]
    
    with torch.no_grad():
        pred_norm = model(
            torch.FloatTensor(hist).unsqueeze(0).to(device),  # [1, WINDOW_SIZE, num_features]
            torch.LongTensor([rid]).to(device),
            torch.LongTensor([oid]).to(device)
        ).cpu().numpy()  # [1, FORECAST_HORIZON]
    
    pred_norm = pred_norm[0]  # [FORECAST_HORIZON]
    
    # Денормализуем
    preds = np.clip(target_scaler.inverse_transform(pred_norm, 'target_2h'), 0, None)
    
    # Соответствуем ID из тестового набора
    for j, original_id in enumerate(batch_ids):
        id_to_pred[original_id] = preds[j]

In [ ]:
submission = pd.DataFrame({
    'id': list(id_to_pred.keys()),
    'y_pred': list(id_to_pred.values())
})
submission = submission.sort_values('id').reset_index(drop=True)

In [ ]:
submission.to_csv('submission_nn_sparse.csv', index=False)
print(f"\nDone! {submission.shape}")
print(f"Sum: {submission['y_pred'].sum():.0f}")
print(submission.head(10))

In [ ]:
model.eval()
vp, vt = [], []
with torch.no_grad(), autocast():
    for xb, rb, ob, yb in val_loader:
        vp.append(model(xb.to(device), rb.to(device), ob.to(device)).cpu().numpy())
        vt.append(yb.numpy())

vp = np.concatenate(vp)  # [N, 10]
vt = np.concatenate(vt)  # [N, 10]

vp_d = np.clip(target_scaler.inverse_transform(vp.flatten(), 'target_2h'), 0, None).reshape(vp.shape)
vt_d = target_scaler.inverse_transform(vt.flatten(), 'target_2h').reshape(vt.shape)
    
wape_plus_rbias(vt_d, vp_d)

In [ ]:
vp.reshape((1,-1))[0].shape

In [ ]:
wape_plus_rbias(vt_d.reshape((1,-1))[0], submission.y_pred.values)

# Диагностика проблемы

### Дефолтная

In [ ]:
model.eval()
vp, vt = [], []
with torch.no_grad():
    for xb, rb, ob, yb in val_loader:
        vp.append(model(xb.to(device), rb.to(device), ob.to(device)).cpu().numpy())
        vt.append(yb.numpy())

vp_d = np.clip(target_scaler.inverse_transform(np.concatenate(vp), 'target_2h'), 0, None)
vt_d = target_scaler.inverse_transform(np.concatenate(vt), 'target_2h')

errors = np.abs(vp_d - vt_d)
rel_errors = errors / (vt_d + 1e-5)

print("Error stats:")
print(f"  MAE: {errors.mean():.4f}")
print(f"  RMSE: {np.sqrt((errors**2).mean()):.4f}")
print(f"  Max error: {errors.max():.4f}")
print(f"  Median error: {np.median(errors):.4f}")
print(f"  90th percentile: {np.percentile(errors, 90):.4f}")
print(f"  % with error > 50: {(errors > 50).mean() * 100:.1f}%")

In [ ]:
val_df = pd.DataFrame({
    'route_id': r_va,
    'true': vt_d,
    'pred': vp_d
})
route_errors = val_df.groupby('route_id').apply(
    lambda g: np.abs(g['pred'] - g['true']).mean()
).sort_values(ascending=False)
print("\nWorst 10 routes:")
print(route_errors.head(10))

In [ ]:
train.groupby('route_id', sort=False)[['status_1', 'status_3']].mean().query('status_1 < 0.5 and status_3 < 0.5').index.tolist()

In [ ]:
val_df['error'] = np.abs(val_df['pred'] - val_df['true'])
val_df['step'] = val_df.groupby('route_id').cumcount()

error_by_step = val_df.groupby('step')['error'].agg(['mean', 'median', 'std'])
print("\nError by prediction step:")
print(error_by_step)

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))
plt.plot(error_by_step.index, error_by_step['mean'], marker='o', label='Mean')
plt.plot(error_by_step.index, error_by_step['median'], marker='s', label='Median')
plt.xlabel('Step (0=first, 9=last)')
plt.ylabel('Absolute Error')
plt.title('Error vs Prediction Step')
plt.legend()
plt.grid()
plt.show()

In [ ]:
worst_routes = [268, 889, 675, 897, 733, 717, 301, 519, 777, 31]

for rid in worst_routes[:3]:
    route_val = val_df[val_df['route_id'] == rid]
    print(f"\nRoute {rid} (MAE={route_val['error'].mean():.1f}):")
#     print(f"  Train samples: {len(train[train['route_id'] == rid])}")
#     print(f"  Val samples:   {len(route_val)}")
    print(f"  Mean target:   {route_val['true'].mean():.1f}")
    print(f"  Mean pred:     {route_val['pred'].mean():.1f}")
    print(f"  Target range:  {route_val['true'].min():.1f} - {route_val['true'].max():.1f}")
    
#     route_train = train[train['route_id'] == rid].copy()
#     route_train['time_slot'] = route_train['timestamp'].dt.hour * 2 + route_train['timestamp'].dt.minute // 30
#     profile = route_train.groupby('time_slot')['target_2h'].agg(['mean', 'std'])
#     print(f"  Target variability (CV): {(profile['std'] / profile['mean']).mean():.2f}")

In [ ]:
problematic = [268, 889, 300, 675]
random_routes = [605, 743, 540] 

for route_list, label in [(problematic, "PROBLEMATIC"), (random_routes, "RANDOM")]:
    print(f"\n{label} ROUTES:")
    for rid in route_list:
        if rid in sample_routes:
            route_data = train[train['route_id'] == rid]
            print(f"\nRoute {rid}:")
            print(f"  Samples: {len(route_data)}")
            print(f"  Target - Mean: {route_data['target_2h'].mean():.1f}, "
                  f"Std: {route_data['target_2h'].std():.1f}, "
                  f"Min: {route_data['target_2h'].min():.1f}, "
                  f"Max: {route_data['target_2h'].max():.1f}")
            print(f"  CoV: {route_data['target_2h'].std() / route_data['target_2h'].mean():.2f}")
            
            hourly = route_data.groupby('hour')['target_2h'].agg(['count', 'mean', 'std'])
            print(f"  Hourly pattern (count/mean/std):")
            print(hourly.to_string())
            
            for status_col in status_cols:
                print(f"  {status_col} - Mean: {route_data[status_col].mean():.1f}, "
                      f"Std: {route_data[status_col].std():.1f}")

In [ ]:
sparse_routes_list = list(sample_routes)[:20]

for rid in sparse_routes_list:
    rd = train[train['route_id'] == rid]
    print(f"\n=== Route {rid} ===")
    print(f"Samples: {len(rd)}")
    print(f"Target mean: {rd['target_2h'].mean():.2f}, std: {rd['target_2h'].std():.2f}")
    print(f"Target min: {rd['target_2h'].min():.2f}, max: {rd['target_2h'].max():.2f}")
    print(f"Office: {rd['office_from_id'].iloc[0]}")
    print("\nStatus stats:")
    for col in status_cols:
        print(f"  {col}: mean={rd[col].mean():.1f}, std={rd[col].std():.1f}, "
              f"min={rd[col].min():.1f}, max={rd[col].max():.1f}")
    
    print("\nCorrelation with target:")
    for col in status_cols:
        corr = rd[[col, 'target_2h']].corr().iloc[0, 1]
        print(f"  {col}: {corr:.3f}")

In [ ]:
sparse_train = train[train['route_id'].isin(sample_routes)]

print("Status usage in sparse routes:")
for col in status_cols:
    non_zero_pct = (sparse_train[col] > 0).mean() * 100
    mean_val = sparse_train[col].mean()
    print(f"{col}: {non_zero_pct:.1f}% non-zero, mean={mean_val:.1f}")

print("\nCorrelation with target (sparse routes only):")
corr_matrix = sparse_train[status_cols + ['target_2h']].corr()
print(corr_matrix['target_2h'].sort_values(ascending=False))

In [ ]:
train_sorted.route_id.unique()

In [ ]:
bad_routes = available_routes

In [ ]:
sparse_train = train_sorted[train_sorted['route_id'].isin(bad_routes)]

model.eval()
vp, vt = [], []
with torch.no_grad(), autocast():
    for xb, rb, ob, yb in val_loader:
        vp.append(model(xb.to(device), rb.to(device), ob.to(device)).cpu().numpy())
        vt.append(yb.numpy())

vp_d = np.clip(target_scaler.inverse_transform(np.concatenate(vp), 'target_2h'), 0, None)
vt_d = target_scaler.inverse_transform(np.concatenate(vt), 'target_2h')

errors = np.abs(vp_d - vt_d)

print("="*70)
print("MODEL_A ERRORS ON SPARSE ROUTES")
print("="*70)

print(f"\nOverall statistics:")
print(f"  MAE: {errors.mean():.2f}")
print(f"  RMSE: {np.sqrt((errors**2).mean()):.2f}")
print(f"  Median error: {np.median(errors):.2f}")
print(f"  Std error: {errors.std():.2f}")
print(f"  Max error: {errors.max():.2f}")


print(f"\nWorst 15 routes by MAE:")
val_df = pd.DataFrame({
    'route_id': r_va,
    'true': vt_d,
    'pred': vp_d
})
route_errors = val_df.groupby('route_id').apply(
    lambda g: np.abs(g['pred'] - g['true']).mean()
).sort_values(ascending=False)
print(route_errors.head(15))

for rid, mae in route_errors.reset_index().values:
    samples = len(sparse_train[sparse_train['route_id']==rid])
    mean_target = val_df[val_df.route_id ==rid]['true'].mean()
    print(f"  Route {rid}: MAE={mae:.2f}, target_mean={mean_target:.2f}, samples={samples}")

print(f"\nError distribution:")
print(f"  % errors < 10: {(errors < 10).mean() * 100:.1f}%")
print(f"  % errors < 20: {(errors < 20).mean() * 100:.1f}%")
print(f"  % errors < 50: {(errors < 50).mean() * 100:.1f}%")
print(f"  % errors >= 50: {(errors >= 50).mean() * 100:.1f}%")

print(f"\nError by target value:")
for threshold in [10, 30, 50, 100, 200]:
    mask = vt_d < threshold
    if mask.sum() > 0:
        print(f"  target < {threshold}: MAE={errors[mask].mean():.2f} ({mask.sum()} samples)")

bias = (vp_d - vt_d).mean()
print(f"\nBias analysis:")
print(f"  Mean bias: {bias:+.2f}")
print(f"  Model tends to {'OVERPREDICT' if bias > 0 else 'UNDERPREDICT'}")

print(f"\nPredictions statistics:")
print(f"  Mean pred: {vp_d.mean():.2f}, Mean true: {vt_d.mean():.2f}")
print(f"  Std pred: {vp_d.std():.2f}, Std true: {vt_d.std():.2f}")
print(f"  Min pred: {vp_d.min():.2f}, Min true: {vt_d.min():.2f}")
print(f"  Max pred: {vp_d.max():.2f}, Max true: {vt_d.max():.2f}")

print("="*70)

In [ ]:
df_normal = pd.read_csv('submission_nn_not_sparse.csv')
df_sparse = pd.read_csv('submission_nn_sparse.csv')
df_ultra_sparse = pd.read_csv('submission_nn_ultra_sparse.csv')

In [ ]:
res = pd.concat([df_normal, df_sparse, df_ultra_sparse], ignore_index=True)
res = res.sort_values('id').reset_index(drop=True)

In [ ]:
res

In [ ]:
# 1. Сравни статистику тренировки
print("="*70)
print("СРАВНЕНИЕ РАЗРЕЖЕННЫХ И НЕРАЗРЕЖЕННЫХ МАРШРУТОВ")
print("="*70)

# Все маршруты с индексом в sparse_routes_list
sparse_mask_train = train['route_id'].isin(available_routes)
dense_mask_train = ~sparse_mask_train

sparse_subset = train[sparse_mask_train]
dense_subset = train[dense_mask_train]

print("\nРАЗРЕЖЕННЫЕ маршруты (в тренировке):")
print(f"  Количество маршрутов: {sparse_subset['route_id'].nunique()}")
print(f"  Количество строк: {len(sparse_subset)}")
print(f"  Target: mean={sparse_subset['target_2h'].mean():.2f}, "
      f"std={sparse_subset['target_2h'].std():.2f}, "
      f"min={sparse_subset['target_2h'].min():.0f}, "
      f"max={sparse_subset['target_2h'].max():.0f}")
print(f"  Коэффициент вариации (std/mean): {sparse_subset['target_2h'].std() / sparse_subset['target_2h'].mean():.2f}")

print("\nНЕРАЗРЕЖЕННЫЕ маршруты (в тренировке):")
print(f"  Количество маршрутов: {dense_subset['route_id'].nunique()}")
print(f"  Количество строк: {len(dense_subset)}")
print(f"  Target: mean={dense_subset['target_2h'].mean():.2f}, "
      f"std={dense_subset['target_2h'].std():.2f}, "
      f"min={dense_subset['target_2h'].min():.0f}, "
      f"max={dense_subset['target_2h'].max():.0f}")
print(f"  Коэффициент вариации: {dense_subset['target_2h'].std() / dense_subset['target_2h'].mean():.2f}")

# 2. Смотри на статусы
print("\n" + "="*70)
print("СРАВНЕНИЕ СТАТУСОВ")
print("="*70)

for col in status_cols:
    sparse_val = sparse_subset[col].mean()
    dense_val = dense_subset[col].mean()
    
    print(f"\n{col}:")
    print(f"  Разреженные: mean={sparse_val:.2f}")
    print(f"  Неразреженные: mean={dense_val:.2f}")
    print(f"  Разница в {dense_val / (sparse_val + 0.001):.1f}x раз")

# 3. Анализ корреляции между статусами и таргетом
print("\n" + "="*70)
print("КОРРЕЛЯЦИЯ (Spearman) СТАТУСОВ С ТАРГЕТОМ")
print("="*70)

from scipy.stats import spearmanr

print("\nРАЗРЕЖЕННЫЕ маршруты:")
for col in status_cols:
    corr, pval = spearmanr(sparse_subset[col], sparse_subset['target_2h'])
    print(f"  {col}: {corr:.4f} (p={pval:.4f})")

print("\nНЕРАЗРЕЖЕННЫЕ маршруты:")
for col in status_cols:
    corr, pval = spearmanr(dense_subset[col], dense_subset['target_2h'])
    print(f"  {col}: {corr:.4f} (p={pval:.4f})")

# 4. Какова доля нулей в каждом статусе?
print("\n" + "="*70)
print("ДОЛЯ НУЛЕЙ В СТАТУСАХ")
print("="*70)

print("\nРАЗРЕЖЕННЫЕ маршруты:")
for col in status_cols:
    zero_pct = (sparse_subset[col] == 0).sum() / len(sparse_subset) * 100
    print(f"  {col}: {zero_pct:.1f}% нулей")

print("\nНЕРАЗРЕЖЕННЫЕ маршруты:")
for col in status_cols:
    zero_pct = (dense_subset[col] == 0).sum() / len(dense_subset) * 100
    print(f"  {col}: {zero_pct:.1f}% нулей")

# 5. Есть ли общие офисы между разреженными и неразреженными?
print("\n" + "="*70)
print("РАСПРЕДЕЛЕНИЕ ПО ОФИСАМ")
print("="*70)

sparse_offices = sparse_subset['office_from_id'].value_counts()
dense_offices = dense_subset['office_from_id'].value_counts()

print(f"\nОфисы в разреженных маршрутах: {sorted(sparse_offices.index.tolist())}")
print(f"Офисы в неразреженных маршрутах: {sorted(dense_offices.index.tolist())}")

common_offices = set(sparse_offices.index) & set(dense_offices.index)
print(f"Общие офисы: {common_offices if common_offices else 'НЕТУ!'}")

In [ ]:
res.to_csv('submission_nn7.csv', index=False)
print(f"\nDone! {res.shape}")
print(f"Sum: {res['y_pred'].sum():.0f}")
print(res.head(10))